# Transfer Learning — Exercises (CIFAR-10 subset)

**Companion to deck 10 (Transfer + Race Day).** Fine-tune ResNet18 on a real image task.

**Critical:** in Colab, switch runtime to GPU (Runtime → Change runtime type → T4 GPU). CPU works but takes 10+ minutes for full training.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/09_transfer_learning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('⚠ no GPU — training will be slow. Switch to GPU runtime if possible.')

## Setup — CIFAR-10 (small subset for speed)

CIFAR-10: 10 classes, 32×32 RGB. We'll resize to 224×224 (ResNet's native input) and use a 2000-image subset to keep training fast.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_train = datasets.CIFAR10('./data', train=True,  download=True, transform=transform)
full_test  = datasets.CIFAR10('./data', train=False, download=True, transform=transform)

# subset for speed — 2000 train, 500 test
train_subset = Subset(full_train, range(2000))
test_subset  = Subset(full_test,  range(500))

CLASSES = ['airplane', 'auto', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print('train:', len(train_subset), '| test:', len(test_subset))
print('classes:', CLASSES)

---
## Problem 01 — load pretrained ResNet18

Load ResNet18 with ImageNet-pretrained weights. Don't change anything yet.

Tasks:
1. `model` — `models.resnet18(pretrained=True)`.
2. Save total parameter count in `total_params`.
3. Save the input feature count of the final FC layer in `fc_in`.

In [ ]:
# TODO
model = ...
total_params = ...
fc_in = ...   # model.fc.in_features

print('total params:', total_params)
print('fc input features:', fc_in)
print('fc output features (ImageNet):', model.fc.out_features)

In [ ]:
assert total_params > 11_000_000, 'ResNet18 should have ~11.7M params'
assert fc_in == 512, f'ResNet18 fc.in_features should be 512, got {fc_in}'
assert model.fc.out_features == 1000, 'pretrained ResNet18 has 1000 ImageNet classes'
print('Q1 ok')

<details><summary>Hint</summary>

Models live in `torchvision.models`. The pretrained flag downloads weights on first run (~45 MB). Total params: sum `p.numel()` over `model.parameters()`. The final layer is named `fc` (a `nn.Linear`). Read its `in_features` attribute.

</details>

---
## Problem 02 — freeze the base, replace the head

Tasks:
1. Loop over all parameters in `model`. Set `requires_grad = False` on each.
2. Replace `model.fc` with a fresh `nn.Linear(fc_in, 10)` (10 classes for CIFAR).
3. Move the model to `device`.
4. Save trainable parameter count in `trainable_params`.

In [ ]:
# TODO
for p in model.parameters():
    ...

model.fc = ...
model = ...   # move to device

trainable_params = ...
print('trainable params:', trainable_params)
print('frozen params:', total_params - trainable_params)

In [ ]:
# new head: 512 * 10 + 10 = 5130
assert trainable_params == 5130, f'expected 5130 trainable params (only the new head), got {trainable_params}'
assert next(model.parameters()).device.type == device
print('Q2 ok — only', trainable_params, 'params train, rest frozen')

<details><summary>Hint</summary>

- Freeze: inside the loop, `p.requires_grad = False`.
- Replace: `model.fc = nn.Linear(fc_in, 10)`. The new layer has `requires_grad=True` by default.
- Move: `model = model.to(device)`.
- Count trainable: sum `p.numel()` over `model.parameters()` filtered by `p.requires_grad`.

</details>

---
## Problem 03 — wire optimizer to the head only

The optimizer should only update the new head's parameters, not the frozen base.

Tasks:
1. `train_loader` — wrap `train_subset` with batch_size=32, shuffle=True.
2. `test_loader` — wrap `test_subset` with batch_size=64, shuffle=False.
3. `optimizer` — Adam with `lr=1e-3`. Pass `model.fc.parameters()` only.

In [ ]:
# TODO
train_loader = ...
test_loader  = ...
criterion = nn.CrossEntropyLoss()
optimizer = ...   # only model.fc.parameters()

print('optimizer param groups:')
for g in optimizer.param_groups:
    print('  params:', sum(p.numel() for p in g['params']), '| lr:', g['lr'])

In [ ]:
n_in_optim = sum(p.numel() for g in optimizer.param_groups for p in g['params'])
assert n_in_optim == 5130, f'optimizer should hold only fc params (5130), got {n_in_optim}'
assert len(train_loader) > 0 and len(test_loader) > 0
print('Q3 ok')

<details><summary>Hint</summary>

Pass `model.fc.parameters()` (not `model.parameters()`) to the Adam constructor. This makes the intent explicit and avoids confusion: the optimizer literally cannot touch the frozen weights.

</details>

---
## Problem 04 — train for 3 epochs

Same 5-line training loop as deck 09. Should converge fast since only 5k params train.

Tasks:
1. Loop 3 epochs over `train_loader`.
2. Run the 5 lines per batch.
3. Save `train_losses` (list of 3 averages, one per epoch).

In [ ]:
# TODO
train_losses = []
for epoch in range(3):
    model.train()
    running = 0.0
    for x, y in train_loader:
        ...   # 5 lines + accumulate loss * batch_size
    avg = running / len(train_subset)
    train_losses.append(avg)
    print(f'epoch {epoch}: loss = {avg:.4f}')

In [ ]:
assert len(train_losses) == 3
assert train_losses[0] > train_losses[-1], 'loss should decrease'
assert train_losses[-1] < 1.5, f'final loss too high: {train_losses[-1]} — check model setup'
print('Q4 ok — final train loss:', round(train_losses[-1], 4))

---
## Problem 05 — evaluate on test set

Tasks:
1. `model.eval()`, `torch.no_grad()`.
2. Predict, count correct, save `test_acc`.
3. Compare to a dummy baseline (always predict class 0).

In [ ]:
# TODO
test_acc = ...

# dummy baseline
from collections import Counter
test_labels = [test_subset[i][1] for i in range(len(test_subset))]
majority = Counter(test_labels).most_common(1)[0][0]
dummy_acc = sum(1 for l in test_labels if l == majority) / len(test_labels)

print('test accuracy:    ', round(test_acc, 3))
print('dummy baseline:   ', round(dummy_acc, 3))
print('lift over dummy:  ', round(test_acc - dummy_acc, 3))

In [ ]:
assert test_acc > 0.5, f'expected >0.5 on CIFAR-10 with frozen ResNet18 + new head, got {test_acc}'
assert test_acc - dummy_acc > 0.30, 'lift over dummy too small — model not learning'
print('Q5 ok — test acc:', round(test_acc, 3))

<details><summary>Hint</summary>

Same eval block as the MNIST notebook:
```
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        ...
```

</details>

---
## Problem 06 — find the bugs in this transfer setup

The cell below tries to set up a transfer model on a new task with 5 classes. It runs but the trainable param count is wrong (10M+ instead of ~3k). Find and fix **three bugs**.

In [ ]:
# BROKEN — fix three things, save trainable param count in `trainable_q6`.
#
# Goal: load pretrained ResNet18, freeze base, replace fc with 5-class head.
#
# Bugs to find:
#  - pretrained=False — random weights, not learned
#  - replace head BEFORE the freeze loop — new head gets frozen too
#  - optimizer is wired to all model parameters — would update frozen weights

model_q6 = models.resnet18(pretrained=False)                            # bug 1

model_q6.fc = nn.Linear(model_q6.fc.in_features, 5)                      # bug 2: BEFORE freeze

for p in model_q6.parameters():
    p.requires_grad = False

optimizer_q6 = optim.Adam(model_q6.parameters(), lr=1e-3)               # bug 3

trainable_q6 = sum(p.numel() for p in model_q6.parameters() if p.requires_grad)
print('trainable params:', trainable_q6)

In [ ]:
# correct: 512 * 5 + 5 = 2565
assert trainable_q6 == 2565, f'expected 2565 trainable params (just the 5-class head), got {trainable_q6}'
n_in_optim_q6 = sum(p.numel() for g in optimizer_q6.param_groups for p in g['params'])
assert n_in_optim_q6 == 2565, f'optimizer should only hold the head, got {n_in_optim_q6} params'
print('Q6 ok')

<details><summary>Hint</summary>

Three bugs:
- `pretrained=False` builds the architecture but skips the learned weights — the whole point of transfer learning is the pretrained values. Set `pretrained=True`.
- Order matters: freeze first, **then** replace the head. If you replace first, the freeze loop touches the new head too. (Or freeze first, swap head, then nothing else needed.)
- Pass `model_q6.fc.parameters()` to the optimizer, not `model_q6.parameters()`. Cleaner intent and matches the head you actually want to train.

</details>

---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Q1
model = models.resnet18(pretrained=True)
total_params = sum(p.numel() for p in model.parameters())
fc_in = model.fc.in_features

# Q2
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(fc_in, 10)
model = model.to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Q3
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_subset,  batch_size=64, shuffle=False)
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

# Q4 — full body
for x, y in train_loader:
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    output = model(x)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    running += loss.item() * x.size(0)

# Q5
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
test_acc = correct / total

# Q6 — fixes
model_q6 = models.resnet18(pretrained=True)         # 1. real weights
for p in model_q6.parameters():                      # 2. freeze FIRST
    p.requires_grad = False
model_q6.fc = nn.Linear(model_q6.fc.in_features, 5)  # then replace
optimizer_q6 = optim.Adam(model_q6.fc.parameters(), lr=1e-3)   # 3. fc only
```
</details>